# Confidence intervals and rank-stability for cell-line recommendations

This notebook produces, for any queried gene, a *confidence interval* on each cell
line's `core_score` and two *rank-stability* measures (how often a line is in the top-10, and how
often it is the single best), by propagating the measurement uncertainty of the scoring inputs through
the scoring function and re-ranking under that uncertainty. It also contains the full verification and
robustness suite used to validate the method.

The notebook is organised as:

1. **What and why** — the problem, the idea, and the design commitments.
2. **Implementation** — constants, the anchor, the scoring chain, and the Monte-Carlo loop.
3. **Real-database wiring** — how a gene's leaves and noise are read from the pipeline outputs.
4. **Visualisations** — the four figures used to interpret the output.
5. **Verification and robustness** — the six checks that validate the intervals.

Everything runs on a self-contained synthetic gene by default; the real-database cells are marked and
require the pipeline outputs plus a DuckDB connection.

## 1 · What we are doing, and why

The scoring stage assigns each cell line a single value $c_{g\ell}\in[0,1]$ — the `core_score` of line
$\ell$ for gene $g$ — and ranks lines by it to recommend those best suited to studying the gene. A
point score, however, conveys no sense of how *trustworthy* a recommendation is: two lines may score
almost identically while one is stable under the inevitable noise in the underlying measurements and
the other would move substantially were those measurements slightly different.

**The idea.** We perturb the scoring inputs by an amount equal to their *measured* uncertainty,
recompute the score, and repeat many times. The spread of the recomputed scores is a confidence
interval; the frequency with which a line remains in the top-$k$ (or is the single best) across
recomputes is its rank-stability. The perturbation sizes are read from disagreement already present in
the data — between the RNA sources, and between the protein platforms — following the logic of
random-effects meta-analysis, rather than being assumed.

**Two design commitments.**

- *The input uncertainties are read from the data, not assumed.* RNA noise comes from the disagreement
  among the three transcriptomic sources; protein noise from the disagreement between the two
  proteomic platforms.
- *The un-perturbed centre reproduces the pipeline's score exactly.* The point score is read from
  `core_score.parquet`; every per-gene constant is recomputed from the gene's own rows using the same
  code as `core_score.py`. This makes the centre draw identical to the pipeline (verified to ≈10⁻⁸),
  so the intervals sit on the real score rather than an approximation.

**Why Monte-Carlo rather than a closed form.** The scoring chain contains a per-`(gene, n_sources)`
standardisation, winsorisation at ±5, a regime switch with a hard `|z|>1` threshold, and a final
normal CDF. The rank-stability outputs are counting functionals of the joint distribution of all
lines' scores, for which no closed form exists; and the regime threshold and winsor clips are
non-smooth, so a first-order (delta-method) propagation is invalid. Simulation is the appropriate
route.

## 2 · Implementation

### 2.1 · Constants (mirroring `core_score.py`) and noise knobs

The first block copies the scoring constants from `core_score.py` so the recompute matches the
pipeline exactly; these must not be changed independently. The second block are the noise knobs — the
only new parameters — which set the perturbation magnitudes. They are deliberately conservative and
are subjected to a sensitivity sweep in §6.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

"""Validated v2 CI logic (re-synced to the rewritten core_score.py). Reused by the demo."""
import numpy as np, pandas as pd
from scipy import stats

W_RNA, W_PROT = np.sqrt(1.431), np.sqrt(1.45)
W_NORM = np.sqrt(W_RNA**2 + W_PROT**2)
RHO_PRIOR, N_MIN_RHO, K_RHO, SD_SHRINK_N0 = 0.353, 30, 30, 25
RNA_STRATUM_MIN_N, RNA_STRATUM_N0, RNA_WINSOR_BOUND = 5, 25, 5.0
PROT_RESID_N0, PROT_RESID_WINSOR_BOUND = 25, 5.0
REGIME3_SIG_THRESH = 1.0
PROT_STD_STRATUM_MIN_N, PROT_STD_STRATUM_N0, PROT_STD_WINSOR_BOUND = 5, 25, 5.0
RNA_BASE_FLOOR, RNA_TAU_PRIOR, COVERAGE_CAP = 0.30, 0.25, 1.5
PROT_BASE, PROT_TAU_PRIOR = 0.30, 0.45
PROT_TIER_MULT = {"consistent": 1.0, "cautious": 1.5, "conflicting": 2.2}
RNA_SOURCES_TOTAL = 3


def _stratum_constants(long_df, value_col, nsrc_col, n0, min_n):
    """Per-stratum location/scale constants with shrinkage toward the global pool.

    Groups `long_df` by source-count stratum (`nsrc_col`) and returns the median
    and standard deviation of `value_col` within each. Thin strata (fewer than
    `min_n` rows) are stabilised by shrinking both their median and variance toward
    the global (pooled) values, with weight w = n / (n + n0); well-populated strata
    keep their raw estimates. This mirrors the within-stratum standardisation used
    by core_score.py so the recompute matches the pipeline.

    Parameters
    ----------
    long_df : pandas.DataFrame
        Long-format leaves for one gene, one row per (model, source-count).
    value_col : str
        Column to standardise (e.g. "rna_z" or "prot_z").
    nsrc_col : str
        Integer source-count column used to define the strata (e.g. "n_sources").
    n0 : int
        Shrinkage prior strength; larger n0 pulls thin strata harder toward global.
    min_n : int
        Minimum stratum size to be treated as "thick" (used raw, unshrunk).

    Returns
    -------
    dict[int, tuple[float, float]]
        Maps each source-count value to (median, sd). Non-finite SDs are floored
        at 1e-6 and fall back to the global SD; non-finite medians fall back to the
        global median.
    """
    med_g = long_df[value_col].median(); sd_g = long_df[value_col].std(ddof=1)
    sd_g = 1e-6 if not np.isfinite(sd_g) else max(sd_g, 1e-6)
    if not np.isfinite(med_g): med_g = 0.0
    out = {}
    for nsrc, sub in long_df.groupby(nsrc_col):
        n = len(sub); med = sub[value_col].median(); sd = sub[value_col].std(ddof=1)
        sd = sd_g if not np.isfinite(sd) else max(sd, 1e-6)
        med = med if np.isfinite(med) else med_g
        w = n/(n+n0); thin = n < min_n
        blended = w*sd**2 + (1-w)*sd_g**2
        out[int(nsrc)] = (float(w*med+(1-w)*med_g) if thin else float(med),
                          float(np.sqrt(blended)) if thin else float(sd))
    return out


def build_anchor(rna_df, prot_df, g):
    """Reconstruct a gene's per-gene scoring constants (the "anchor").

    Recomputes, from the gene's own leaves, every constant core_score.py needs to
    rebuild the score: the RNA and protein per-stratum medians/SDs, the RNA→protein
    regression slope beta_g, and the protein-residual median/SD. beta_g and the
    stratum constants are exactly recoverable per gene because each is
    column-independent; the three panel-wide scalars that are not (global RNA,
    protein and residual SD) are supplied via `g`.

    The slope is estimated as beta_g = rho_g * (sd_prot / sd_rna), where rho_g is a
    Pearson correlation on RNA/protein-shared lines shrunk toward RHO_PRIOR (weight
    K_RHO / (n + K_RHO)) and the SDs are shrunk toward their global values. If fewer
    than N_MIN_RHO shared lines exist, beta_g falls back to the prior slope
    RHO_PRIOR * (G_PROT / G_RNA).

    Parameters
    ----------
    rna_df : pandas.DataFrame
        RNA leaves for the gene; requires columns "model_id", "rna_z", "n_sources".
    prot_df : pandas.DataFrame
        Protein leaves for the gene; requires "model_id", "prot_z", "n_sources".
        May be empty for RNA-only genes.
    g : dict
        Panel-wide global scalars persisted by the pipeline to ci_globals.json:
        "sd_rna_global", "sd_prot_global", "prot_resid_sd_global".

    Returns
    -------
    dict
        Anchor with keys: "rna_str", "protstd_str" (stratum → (median, sd) maps),
        "beta_g" (float slope), "resid_med", "resid_sd" (protein-residual location
        and scale). Consumed by `core_score_chain`.
    """
    G_RNA, G_PROT, G_RESID = g["sd_rna_global"], g["sd_prot_global"], g["prot_resid_sd_global"]
    rna_str = _stratum_constants(rna_df, "rna_z", "n_sources", RNA_STRATUM_N0, RNA_STRATUM_MIN_N)
    protstd_str = (_stratum_constants(prot_df, "prot_z", "n_sources",
                   PROT_STD_STRATUM_N0, PROT_STD_STRATUM_MIN_N) if len(prot_df) else {})
    shared = prot_df.merge(rna_df[["model_id", "rna_z"]], on="model_id", how="inner")
    if len(shared) >= N_MIN_RHO:
        rho_raw = stats.pearsonr(shared.prot_z.fillna(0), shared.rna_z.fillna(0))[0]
        n = len(shared); lam = K_RHO/(n+K_RHO); rho_g = lam*RHO_PRIOR + (1-lam)*rho_raw
        w_sd = n/(n+SD_SHRINK_N0)
        sd_rna = np.sqrt(w_sd*shared.rna_z.std(ddof=1)**2 + (1-w_sd)*G_RNA**2)
        sd_prot = np.sqrt(w_sd*shared.prot_z.std(ddof=1)**2 + (1-w_sd)*G_PROT**2)
        beta_g = rho_g*(sd_prot/sd_rna) if sd_rna > 0 else rho_g
    else:
        beta_g = RHO_PRIOR*(G_PROT/G_RNA) if G_RNA > 0 else RHO_PRIOR
    if len(shared):
        resid = shared.prot_z - beta_g*shared.rna_z; n = len(shared)
        sd_g = max(resid.std(ddof=1), 1e-6); w = n/(n+PROT_RESID_N0)
        resid_med, resid_sd = float(resid.median()), float(np.sqrt(w*sd_g**2 + (1-w)*G_RESID**2))
    else:
        beta_g, resid_med, resid_sd = 0.0, 0.0, 1.0
    return {"rna_str": rna_str, "protstd_str": protstd_str,
            "beta_g": float(beta_g), "resid_med": resid_med, "resid_sd": resid_sd}


def _lookup(nsrc, strata, default=(0.0, 1.0)):
    """Vectorised stratum lookup of (median, sd) by source count.

    Parameters
    ----------
    nsrc : array-like
        Source counts per line; non-finite entries use `default`.
    strata : dict[int, tuple[float, float]]
        Stratum → (median, sd) map, e.g. an anchor's "rna_str" or "protstd_str".
    default : tuple[float, float], optional
        Fallback (median, sd) for missing or non-finite strata. Defaults to (0.0, 1.0),
        i.e. an identity standardisation.

    Returns
    -------
    tuple[numpy.ndarray, numpy.ndarray]
        (medians, sds), each aligned element-wise with `nsrc`.
    """
    med = np.array([strata.get(int(k), default)[0] if np.isfinite(k) else default[0] for k in nsrc])
    sd = np.array([strata.get(int(k), default)[1] if np.isfinite(k) else default[1] for k in nsrc])
    return med, sd


def core_score_chain(rna_z, prot_z, nsrc_rna, nsrc_prot, has_p, a):
    """Recompute the per-line core_score from RNA/protein z-scores and an anchor.

    Reproduces core_score.py exactly. RNA-only lines use the standardised,
    winsorised RNA arm alone. Two-layer lines combine the RNA arm with a protein
    arm under one of two regimes:

    - Regime A (consistent): protein enters as its RNA-residual r = z_P - beta_g z_R,
      standardised and winsorised — i.e. only the part of protein not predictable
      from RNA contributes.
    - Regime C (conflicting): when both standardised signals exceed REGIME3_SIG_THRESH
      in magnitude and disagree in sign, protein enters as its own standardised z
      instead of the residual, so a genuine RNA/protein conflict is not partly
      cancelled by the regression.

    Both arms are winsorised at ±5 and combined with fixed weights W_RNA, W_PROT
    (normalised by W_NORM). The combined z is mapped to [0, 1] via the standard
    normal CDF.

    Parameters
    ----------
    rna_z, prot_z : numpy.ndarray
        Per-line RNA and protein z-scores (prot_z may be NaN where absent).
    nsrc_rna, nsrc_prot : numpy.ndarray
        Per-line source counts, used to select the standardisation stratum.
    has_p : numpy.ndarray of bool
        True where a protein layer is present.
    a : dict
        Anchor from `build_anchor`.

    Returns
    -------
    numpy.ndarray
        core_score in [0, 1] per line (Phi of the combined z).
    """
    rmed, rsd = _lookup(nsrc_rna, a["rna_str"])
    rna_std_z = np.clip((rna_z - rmed)/rsd, -RNA_WINSOR_BOUND, RNA_WINSOR_BOUND)
    core_z = np.where(has_p, np.nan, (W_RNA/W_NORM)*rna_std_z)
    if has_p.any():
        pmed, psd = _lookup(np.where(has_p, nsrc_prot, np.nan), a["protstd_str"])
        with np.errstate(invalid="ignore"):
            prot_std_z = np.clip((prot_z - pmed)/psd, -PROT_STD_WINSOR_BOUND, PROT_STD_WINSOR_BOUND)
            prot_resid = prot_z - a["beta_g"]*rna_z
            prot_resid_z = np.clip((prot_resid - a["resid_med"])/a["resid_sd"],
                                   -PROT_RESID_WINSOR_BOUND, PROT_RESID_WINSOR_BOUND)
            core_z_A = (W_RNA*rna_std_z + W_PROT*prot_resid_z)/W_NORM
            core_z_C = (W_RNA*rna_std_z + W_PROT*np.nan_to_num(prot_std_z))/W_NORM
            reg3 = ((np.abs(rna_std_z) > REGIME3_SIG_THRESH) & (np.abs(prot_std_z) > REGIME3_SIG_THRESH)
                    & (np.sign(rna_std_z) != np.sign(prot_std_z)) & np.isfinite(prot_std_z))
        core_z = np.where(has_p, np.where(reg3, core_z_C, core_z_A), core_z)
    return stats.norm.cdf(core_z)


def monte_carlo_ci(base, anchor, N=1000, k=10, seed=42, ci=0.95):
    """Propagate input measurement noise through scoring to get CIs and rank stability.

    Draws N Monte-Carlo replicates in which each line's RNA and protein z-scores are
    perturbed by independent Gaussian noise scaled to their standard errors
    (`se_rna`, `se_prot`), re-scores and re-ranks under each draw, and summarises the
    resulting score distribution and ranking behaviour. The centre (unperturbed)
    score is computed once and the CI is clamped to always contain it.

    Parameters
    ----------
    base : pandas.DataFrame
        One row per line, with columns "model_id", "lineage", "rna_z", "se_rna",
        "n_sources_rna", "prot_z", "se_prot", "n_sources_prot".
    anchor : dict
        Per-gene constants from `build_anchor`.
    N : int, optional
        Number of Monte-Carlo draws (default 1000).
    k : int, optional
        Top-k cutoff for the stability frequencies (default 10).
    seed : int, optional
        RNG seed for reproducibility (default 42).
    ci : float, optional
        Central coverage of the interval, e.g. 0.95 for a 95% CI.

    Returns
    -------
    tuple[pandas.DataFrame, numpy.ndarray, numpy.ndarray]
        - Results table sorted by `core_score`, with `ci_lo`, `ci_hi`, `ci_width`,
          `topk_global_pct` (top-k frequency against all lines), `topk_lineage_pct`
          (top-k within the line's lineage), `top1_winrate` (single-best frequency),
          `n_layers` and `n_sources`.
        - `scores`: the (N, n) matrix of simulated scores.
        - `ids`: the model_id order matching `scores` columns (pre-sort).
    """
    rng = np.random.default_rng(seed)
    ids = base.model_id.to_numpy(); n = len(base); lineage = base.lineage.to_numpy()
    rna_mu, rna_sd = base.rna_z.to_numpy(), base.se_rna.to_numpy()
    nsrc_rna = base.n_sources_rna.to_numpy(float)
    has_p = base.prot_z.notna().to_numpy()
    prot_mu = base.prot_z.to_numpy(float); prot_sd = np.nan_to_num(base.se_prot.to_numpy(float))
    nsrc_prot = base.n_sources_prot.to_numpy(float)
    score = lambda rz, pz: core_score_chain(rz, pz, nsrc_rna, nsrc_prot, has_p, anchor)
    centre = score(rna_mu, prot_mu)
    scores = np.empty((N, n)); top_g = np.zeros(n); top_l = np.zeros(n); win1 = np.zeros(n)
    for i in range(N):
        rz = rna_mu + rng.normal(0, 1, n)*rna_sd
        pz = prot_mu.copy(); pz[has_p] = prot_mu[has_p] + rng.normal(0, 1, int(has_p.sum()))*prot_sd[has_p]
        s = score(rz, pz); scores[i] = s
        o = np.argsort(-s); top_g[o[:k]] += 1; win1[o[0]] += 1
        t = pd.DataFrame({"s": s, "lin": lineage})
        t["r"] = t.groupby("lin")["s"].rank(ascending=False, method="first")
        top_l[(t["r"] <= k).to_numpy()] += 1
    lo, hi = (1-ci)/2, 1-(1-ci)/2
    out = pd.DataFrame({"model_id": ids, "lineage": lineage, "core_score": centre,
        "ci_lo": np.quantile(scores, lo, axis=0), "ci_hi": np.quantile(scores, hi, axis=0),
        "topk_global_pct": 100*top_g/N, "topk_lineage_pct": 100*top_l/N, "top1_winrate": 100*win1/N,
        "n_layers": np.where(has_p, 2, 1), "n_sources": base.n_sources_rna.to_numpy()})
    out["ci_lo"] = np.minimum(out.ci_lo, out.core_score); out["ci_hi"] = np.maximum(out.ci_hi, out.core_score)
    out["ci_width"] = out.ci_hi - out.ci_lo
    return out.sort_values("core_score", ascending=False).reset_index(drop=True), scores, ids


### 2.2 · What the chain computes

For a queried gene, the score is reconstructed per line as follows (this mirrors the current
`core_score.py`):

- **RNA arm.** Standardise within the line's `(gene, n_sources)` stratum and winsorise at ±5:
  $\;\tilde z^R_{g\ell}=\mathrm{clip}\!\big((z^R_{g\ell}-m^R)/s^R,\ \pm 5\big)$, where $m^R,s^R$ are the
  stratum median and (thin-stratum-shrunk) standard deviation.
- **Protein residual arm.** Remove the RNA-predictable part and standardise:
  $\;r_{g\ell}=z^P_{g\ell}-\beta_g z^R_{g\ell}$, then
  $\tilde r_{g\ell}=\mathrm{clip}\!\big((r_{g\ell}-m^r)/s^r,\ \pm5\big)$.
- **Combination.** $\;\zeta=(w_R\tilde z^R+w_P\tilde r)/\sqrt{w_R^2+w_P^2}$ for two-layer lines;
  RNA-only lines use $\zeta=(w_R/\sqrt{w_R^2+w_P^2})\,\tilde z^R$ (a deliberate down-weight). The final
  score is $c=\Phi(\zeta)$.
- **Regime substitution.** For two-layer lines where RNA and protein are both strong
  ($|\tilde z^R|>1$ and $|\tilde z^P|>1$) and point in opposite directions, the combination uses the
  independently-standardised protein value $\tilde z^P$ in place of the residual.

The weights $w_R^2=1.431$ and $w_P^2=1.45$ are effective sample sizes discounting the three RNA
sources and two protein platforms for their mutual correlation.

### 2.3 · Anchoring — reproducing the pipeline's per-gene constants

The per-gene constants ($\beta_g$, the residual median/SD, and the per-stratum medians/SDs) are
recomputed from the gene's own rows, which is exact because each is column-independent. Three
panel-wide scalars — the global RNA SD, protein SD, and protein-residual SD — cannot be recovered from
a single gene, so `core_score.py` persists them once to `ci_globals.json`:

```python
# added to core_score.py's run(), after prot_resid_wide is built:
import json
(CORE_SCORE.parent / "ci_globals.json").write_text(json.dumps({
    "sd_rna_global":  float(_sd_rna_global),
    "sd_prot_global": float(_sd_prot_global),
    "prot_resid_sd_global": float(prot_resid_wide.stack().std(ddof=1)),
}))
```

`build_anchor` (defined above) reads those three scalars and reconstructs everything else per gene.

## 3 · Real-database wiring

These functions read a gene's leaves and their noise from the pipeline outputs. The RNA and protein
*disagreement* — which sizes the perturbation — is read directly from the per-source exports
(`bulk_rna_z_by_source.parquet`, `bulk_prot_z_by_source.parquet`), so the noise model uses exactly the
per-source values that fed the combined score. Single-source / single-platform lines, having no
observable disagreement, fall back to a corpus prior (RNA) or the gene-level platform tier (protein).

These cells require the pipeline environment (`config`, `common`, `protein_scorer`) on the path and a
DuckDB connection; they are defined here and called in the real-data section.

In [ ]:
def _resolve_gid(con, gene):
    """Resolve a gene symbol or Ensembl ID to a canonical upper-case gene_id.

    Passes through inputs already starting with "ENSG"; otherwise looks up the HUGO
    symbol (case-insensitive) in the `gene` table.

    Parameters
    ----------
    con : database connection
        Open warehouse connection (DuckDB).
    gene : str
        A HUGO symbol (e.g. "EGFR") or an Ensembl gene ID.

    Returns
    -------
    str
        Upper-case Ensembl gene_id.

    Raises
    ------
    ValueError
        If the symbol is not found.
    """
    g = str(gene).strip()
    if g.upper().startswith("ENSG"): return g.upper()
    r = con.execute("SELECT gene_id FROM gene WHERE lower(hugo_symbol)=lower(?) LIMIT 1", [g]).fetchone()
    if not r: raise ValueError(f"gene not found: {gene}")
    return r[0].upper()


def _gene_leaves(con, gene):
    """Read a gene's per-line RNA/protein point scores and noise from pipeline outputs.

    Loads the RNA and protein z-scores for the gene and sizes their standard errors
    from the *measured* between-source disagreement exported per source, so the noise
    model uses exactly the per-source values that fed the combined score:

    - RNA: `se_rna` combines a between-source term (the SD across DepMap/HPA/GEO
      divided by sqrt(k), falling back to RNA_TAU_PRIOR when unavailable) with a
      coverage-scaled within-source floor.
    - Protein: `se_prot` combines the gene's platform-tier floor (from PROT_TIER,
      mapped through PROT_TIER_MULT) with the measured ProCAN-vs-CCLE disagreement.
      Single-platform lines, having no observable disagreement, use a prior instead.

    Parameters
    ----------
    con : database connection
        Open warehouse connection.
    gene : str
        Gene symbol or Ensembl ID (resolved via `_resolve_gid`).

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame, str]
        (rz, pz, gid) — RNA leaves with columns "model_id", "rna_z", "n_sources",
        "se_rna"; protein leaves with "model_id", "prot_z", "n_sources", "se_prot";
        and the resolved gene_id. model_id is lower-cased in both frames.
    """
    import protein_scorer as PS
    from config import RNA_Z, PROT_Z, PROT_TIER
    gid = _resolve_gid(con, gene).upper()

    # point scores (note: RNA source-count column is k_src in this pipeline build)
    rz = pd.read_parquet(RNA_Z).rename(columns={"z_t": "rna_z", "k_src": "n_sources"})
    rz["gene_id"] = rz.gene_id.str.upper()
    rz = rz[rz.gene_id == gid][["model_id", "rna_z", "n_sources"]].copy(); rz["model_id"] = rz.model_id.str.lower()
    pz = pd.read_parquet(PROT_Z).rename(columns={"z_t": "prot_z"})
    pz["gene_id"] = pz.gene_id.str.upper()
    pz = pz[pz.gene_id == gid][["model_id", "prot_z", "n_sources"]].copy(); pz["model_id"] = pz.model_id.str.lower()

    # RNA between-source disagreement, read directly from the per-source export
    rbs = pd.read_parquet(RNA_Z.parent / "bulk_rna_z_by_source.parquet")
    rbs["gene_id"] = rbs.gene_id.str.upper(); rbs = rbs[rbs.gene_id == gid].copy()
    rbs["model_id"] = rbs.model_id.str.lower()
    rbs["s_between"] = rbs[["z_depmap", "z_hpa_rna", "z_geo"]].std(axis=1, ddof=1)
    rz = rz.merge(rbs[["model_id", "s_between"]], on="model_id", how="left")
    k = rz.n_sources.to_numpy()
    se_b = np.where(k >= 2, rz.s_between.to_numpy()/np.sqrt(np.maximum(k, 1)), RNA_TAU_PRIOR)
    se_b = np.where(np.isfinite(se_b), se_b, RNA_TAU_PRIOR)
    se_w = RNA_BASE_FLOOR*np.minimum(np.sqrt(RNA_SOURCES_TOTAL/np.maximum(k, 1)), COVERAGE_CAP)
    rz["se_rna"] = np.sqrt(se_b**2 + se_w**2)

    # protein noise: gene-level platform tier as floor + measured ProCAN-vs-CCLE disagreement
    u2e = PS._build_uniprot_ensg(con); uni = {u for u, e in u2e.items() if e == gid}
    tier_df = pd.read_parquet(PROT_TIER); tier_df["uniprot"] = tier_df.uniprot.str.lower()
    trow = tier_df[tier_df.uniprot.isin(uni)]
    tier = str(trow.sort_values("platform_rho").iloc[0].platform_tier) if len(trow) else "cautious"
    mult = PROT_TIER_MULT.get(tier, 1.5)
    pbs = pd.read_parquet(PROT_Z.parent / "bulk_prot_z_by_source.parquet")
    pbs["gene_id"] = pbs.gene_id.str.upper(); pbs = pbs[pbs.gene_id == gid].copy()
    pbs["model_id"] = pbs.model_id.str.lower()
    pbs["p_between"] = pbs[["z_procan", "z_ccle"]].std(axis=1, ddof=1)/np.sqrt(2)
    pz = pz.merge(pbs[["model_id", "p_between"]], on="model_id", how="left")
    tier_floor = mult*PROT_BASE; p_between = pz.p_between.to_numpy(); one_plat = pz.n_sources.to_numpy() < 2
    se_meas = np.sqrt(np.nan_to_num(p_between)**2 + tier_floor**2)
    se_prior = np.sqrt(tier_floor**2 + PROT_TAU_PRIOR**2)
    pz["se_prot"] = np.where(one_plat | ~np.isfinite(p_between), se_prior, se_meas)
    return rz, pz, gid


def run_gene_ci(con, gene, N=1000, k=10, seed=42):
    """End-to-end confidence intervals and rank stability for one gene, from the warehouse.

    Reads the gene's leaves and noise (`_gene_leaves`), loads the panel-wide globals
    from ci_globals.json, builds the per-gene anchor (`build_anchor`), attaches
    lineage, assembles the `base` frame, and runs the Monte-Carlo propagation.

    Parameters
    ----------
    con : database connection
        Open warehouse connection.
    gene : str
        Gene symbol or Ensembl ID.
    N : int, optional
        Monte-Carlo draws (default 1000).
    k : int, optional
        Top-k cutoff for stability frequencies (default 10).
    seed : int, optional
        RNG seed (default 42).

    Returns
    -------
    tuple[pandas.DataFrame, numpy.ndarray, numpy.ndarray]
        As `monte_carlo_ci`: the sorted results table, the (N, n) score matrix,
        and the model_id order for that matrix.
    """
    import json
    from config import CORE_SCORE
    import common as C
    rz, pz, gid = _gene_leaves(con, gene)
    g_globals = json.loads((CORE_SCORE.parent/"ci_globals.json").read_text())
    anchor = build_anchor(rz, pz, g_globals)
    lm = C.load_lineage(con); lm.index = lm.index.str.lower()
    base = (rz.rename(columns={"n_sources": "n_sources_rna"})
              .merge(pz.rename(columns={"n_sources": "n_sources_prot"}), on="model_id", how="left"))
    base["lineage"] = base.model_id.map(lm)
    return monte_carlo_ci(base, anchor, N=N, k=k, seed=seed)

### 3.1 · Running on the warehouse and the centre-draw check

The single most important verification: with the noise turned off, the recomputed score must equal the
stored `core_score`. A clean result is a maximum absolute deviation of order $10^{-6}$ or smaller — in
our runs $\approx 1.8\times10^{-8}$ across EGFR, TP53 and MYC. The cell below is left commented so the
notebook runs without a database.

In [ ]:
import sys
from pathlib import Path
PIPELINE_ROOT = Path("/Users/musa.official/Documents/UoB-GeneTraceAI-25-26/final_pipeline").resolve()
for p in (PIPELINE_ROOT, PIPELINE_ROOT/"utils", PIPELINE_ROOT/"02_Proteinomics", PIPELINE_ROOT/"Scoring"):
    if str(p) not in sys.path: sys.path.insert(0, str(p))

In [ ]:
import duckdb
from config import DB, CORE_SCORE
con = duckdb.connect(str(DB), read_only=True)
res, scores, ids = run_gene_ci(con, "EGFR", N=1000, k=10)

core = pd.read_parquet(CORE_SCORE); core["ensg_id"] = core.ensg_id.str.upper()
gid = _resolve_gid(con, "EGFR").upper()
ref = core[core.ensg_id==gid].assign(model_id=lambda d: d.model_id.str.lower())
got = res.assign(model_id=lambda d: d.model_id.str.lower())
m = got.merge(ref[["model_id","core_score"]].rename(columns={"core_score":"r"}), on="model_id")
print("compared:", len(m), "| max|Δ|:", (m.core_score - m.r).abs().max())    # -> ~1.8e-08

In [ ]:
id_ix = {m: i for i, m in enumerate(ids)}

print(f"lines: {len(res)}")
print(f"with protein: {res['n_layers'].eq(2).sum()}")
print(f"RNA sources: {res['n_sources'].value_counts().to_dict()}")

display(
    res.head(10)[[
        "model_id",
        "lineage",
        "core_score",
        "ci_lo",
        "ci_hi",
        "topk_global_pct",
        "topk_lineage_pct",
        "top1_winrate",
        "n_layers",
        "n_sources"
    ]].round(3)
)

### How to read the output

- **`core_score`** — the fit score (0–1), identical to the pipeline. Near the top it saturates toward
  1.0 (it is $\Phi$ of a combined $z$), so the interval and win-rate carry information the point score
  cannot.
- **`ci_lo, ci_hi, ci_width`** — the 95% interval and its width. Wide means the score would move a lot
  under realistic measurement noise.
- **`top1_winrate`** — how often (%) this line is the single best across all lines; the honest
  "is it *the* best?" number.
- **`topk_global_pct` vs `topk_lineage_pct`** — top-10 frequency against all lines vs against
  same-tissue lines. A line can be stable within its tissue yet contested globally.
- **`n_layers`** — 2 if both RNA and protein, 1 if RNA only. **`n_sources`** — number of RNA sources.

## 4 · Visualisations

**Figure 1 — recommendations with 95% intervals.** Each line's score (dot) and interval (bar), best at top, coloured by evidence layers, with the top-1 win-rate at right. Overlap near the top shows the ranking is genuinely uncertain there.

In [ ]:
plt.rcParams.update({"figure.dpi":120,"font.size":10,"axes.spines.top":False,
                     "axes.spines.right":False,"axes.grid":True,"grid.alpha":.25,"axes.axisbelow":True})
INK, C2, C1 = "#1b2a4a", "#2f6db3", "#c05a3a"
top = res.head(20).iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9,7))
for i, r in top.iterrows():
    col = C2 if r.n_layers==2 else C1
    ax.plot([r.ci_lo, r.ci_hi],[i,i], color=col, lw=2.4, alpha=.55, solid_capstyle="round")
    ax.scatter(r.core_score, i, color=col, s=70, zorder=3, edgecolor="white", lw=1)
    ax.text(1.006, i, f"{r.top1_winrate:4.0f}%", va="center", ha="left", fontsize=8.5, family="monospace", color=INK)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.model_id, fontsize=8.5)
ax.set_xlabel("core_score  (point + 95% CI)"); ax.set_xlim(0.4, 1.005)
ax.set_title("Best-fit cell lines — score with confidence interval", fontweight="bold", color=INK, loc="left")
ax.legend(handles=[Patch(color=C2,label="RNA + protein"),Patch(color=C1,label="RNA only")],
          loc="lower left", frameon=False); plt.tight_layout(); plt.show()

**Figure 2 — same uncertainty, two rankings.** Global vs within-tissue top-10 stability. Points high on the left are stable within their tissue but contested globally — typically leaders of a crowded tissue. Same noise, different competitive neighbourhood.

In [ ]:
d = res[res.core_score > 0.5]
fig, ax = plt.subplots(figsize=(7.4,6.2))
sc = ax.scatter(d.topk_global_pct, d.topk_lineage_pct, c=d.core_score, cmap="viridis",
                s=46, edgecolor="white", lw=.6)
ax.plot([0,100],[0,100],"--",color="gray",lw=1,alpha=.6)
ax.set_xlabel("global top-10 stability (%)"); ax.set_ylabel("within-tissue top-10 stability (%)")
ax.set_title("Same uncertainty, two rankings", fontweight="bold", color=INK, loc="left")
fig.colorbar(sc, label="core_score", shrink=.85); plt.tight_layout(); plt.show()

**Figure 3 — interval width vs source coverage.** Width is not monotone in coverage: single-source lines are widest (prior fallback), and three-source lines can be wider than two-source ones when the third source contributes genuine disagreement. Width tracks measured disagreement, not source count.

In [ ]:
fig, ax = plt.subplots(figsize=(6.6,5.0))
groups = [res[res.n_sources==kk].ci_width.values for kk in (1,2,3)]; groups=[x for x in groups if len(x)]
parts = ax.violinplot(groups, positions=range(1,len(groups)+1), showmedians=True, widths=.8)
for b in parts["bodies"]: b.set_facecolor(C2); b.set_alpha(.35); b.set_edgecolor(INK)
for key in ("cmedians","cbars","cmins","cmaxes"): parts[key].set_color(INK)
means=[x.mean() for x in groups]; ax.plot(range(1,len(groups)+1), means, "o-", color=C1, lw=2, label="mean width")
for x,mu in zip(range(1,len(groups)+1),means): ax.text(x,mu+.02,f"{mu:.2f}",ha="center",color=C1,fontweight="bold",fontsize=9)
ax.set_xticks(range(1,len(groups)+1)); ax.set_xlabel("number of RNA sources"); ax.set_ylabel("95% CI width")
ax.set_title("Interval width vs source coverage", fontweight="bold", color=INK, loc="left")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

**Figure 4 — what the interval is.** The Monte-Carlo score distribution for a confident line (tight spike) and a contested one (broad spread). The confidence interval is the central 95% of the simulated scores.

In [ ]:
plt.rcParams.update({"figure.dpi":120,"font.size":10,"axes.spines.top":False,
                     "axes.spines.right":False,"axes.grid":True,"grid.alpha":.25,
                     "axes.axisbelow":True,"axes.facecolor":"white","figure.facecolor":"white"})
INK, C2, C1 = "#1b2a4a", "#2f6db3", "#c05a3a"

confident = res.iloc[0].model_id
# contested = widest interval among lines that are NOT the confident one and not saturated
cand = res[(res.model_id != confident) & (res.core_score < 0.97)].copy()
contested = cand.sort_values("ci_width", ascending=False).iloc[0].model_id
assert contested != confident, "picker still returned the confident line"

fig, ax = plt.subplots(figsize=(7.6,4.6))
for mid, col, lab in [(confident, C2, "confident"), (contested, C1, "contested")]:
    s = scores[:, id_ix[mid]]
    ax.hist(s, bins=40, color=col, alpha=.5, density=True, label=f"{mid} ({lab})")
    r = res.set_index("model_id").loc[mid]
    ax.axvline(r.core_score, color=col, lw=2); ax.axvspan(r.ci_lo, r.ci_hi, color=col, alpha=.08)
ax.set_xlabel("core_score across simulation runs"); ax.set_ylabel("density")
ax.set_title("What the interval is: spread of the score under noise", fontweight="bold", color=INK, loc="left")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

print("confident:", confident, "| contested:", contested)   # confirm two different real ach-000xxx IDs

## CI Checks

In [ ]:
def monte_carlo_ci_corr(base, anchor, N=1000, k=10, seed=42, ci=0.95, rho_err=0.0):
    """Monte-Carlo CIs with correlated RNA/protein measurement errors.

    Identical to `monte_carlo_ci` except the protein perturbation is drawn with
    correlation `rho_err` to the RNA perturbation, via
    e_prot = rho * e_rna + sqrt(1 - rho^2) * e_ind. Used to test whether the
    baseline's independent-error assumption is load-bearing; rho_err=0 reproduces
    that baseline. All outputs are returned, though only the win-rate / stability
    columns are used downstream.

    Parameters
    ----------
    base : pandas.DataFrame
        Per-line frame as in `monte_carlo_ci`.
    anchor : dict
        Per-gene constants from `build_anchor`.
    N, k, seed, ci : optional
        As in `monte_carlo_ci`.
    rho_err : float, optional
        Target error correlation, clipped to (-0.999, 0.999). Default 0.0.

    Returns
    -------
    pandas.DataFrame
        Results table sorted by `core_score`, with the same columns as
        `monte_carlo_ci`'s table (the score matrix and id array are not returned).
    """
    rng = np.random.default_rng(seed)
    ids = base.model_id.to_numpy(); n = len(base); lineage = base.lineage.to_numpy()
    rna_mu, rna_sd = base.rna_z.to_numpy(), base.se_rna.to_numpy()
    nsrc_rna = base.n_sources_rna.to_numpy(float)
    has_p = base.prot_z.notna().to_numpy()
    prot_mu = base.prot_z.to_numpy(float); prot_sd = np.nan_to_num(base.se_prot.to_numpy(float))
    nsrc_prot = base.n_sources_prot.to_numpy(float)
    a = anchor
    score = lambda rz, pz: core_score_chain(rz, pz, nsrc_rna, nsrc_prot, has_p, a)

    centre = score(rna_mu, prot_mu)
    scores = np.empty((N, n)); top_g = np.zeros(n); top_l = np.zeros(n); win1 = np.zeros(n)
    rho = float(np.clip(rho_err, -0.999, 0.999))
    for i in range(N):
        e_rna = rng.normal(0, 1, n)
        e_ind = rng.normal(0, 1, n)
        e_prot = rho * e_rna + np.sqrt(1 - rho**2) * e_ind      # correlated standard-normal errors
        rz = rna_mu + e_rna * rna_sd
        pz = prot_mu.copy()
        pz[has_p] = prot_mu[has_p] + e_prot[has_p] * prot_sd[has_p]
        s = score(rz, pz); scores[i] = s
        o = np.argsort(-s); top_g[o[:k]] += 1; win1[o[0]] += 1
        t = pd.DataFrame({"s": s, "lin": lineage})
        t["r"] = t.groupby("lin")["s"].rank(ascending=False, method="first")
        top_l[(t["r"] <= k).to_numpy()] += 1

    lo, hi = (1-ci)/2, 1-(1-ci)/2
    out = pd.DataFrame({"model_id": ids, "lineage": lineage, "core_score": centre,
        "ci_lo": np.quantile(scores, lo, axis=0), "ci_hi": np.quantile(scores, hi, axis=0),
        "topk_global_pct": 100*top_g/N, "topk_lineage_pct": 100*top_l/N, "top1_winrate": 100*win1/N,
        "n_layers": np.where(has_p, 2, 1), "n_sources": base.n_sources_rna.to_numpy()})
    out["ci_lo"] = np.minimum(out.ci_lo, out.core_score); out["ci_hi"] = np.maximum(out.ci_hi, out.core_score)
    out["ci_width"] = out.ci_hi - out.ci_lo
    return out.sort_values("core_score", ascending=False).reset_index(drop=True)


def _base_anchor(con, gene):
    """Build the (base, anchor) pair for a gene — the wiring `run_gene_ci` uses internally.

    Reads the gene's leaves and noise, loads the panel-wide globals, builds the
    anchor, attaches lineage, and assembles the `base` frame — stopping short of the
    Monte-Carlo step so callers (e.g. the sensitivity sweep) can drive the simulation
    themselves.

    Parameters
    ----------
    con : database connection
        Open warehouse connection.
    gene : str
        Gene symbol or Ensembl ID.

    Returns
    -------
    tuple[pandas.DataFrame, dict]
        (base, anchor), ready to pass to `monte_carlo_ci` or `monte_carlo_ci_corr`.
    """
    import json
    from config import CORE_SCORE
    import common as C
    rz, pz, gid = _gene_leaves(con, gene)
    g_globals = json.loads((CORE_SCORE.parent/"ci_globals.json").read_text())
    anchor = build_anchor(rz, pz, g_globals)
    lm = C.load_lineage(con); lm.index = lm.index.str.lower()
    base = (rz.rename(columns={"n_sources":"n_sources_rna"})
              .merge(pz.rename(columns={"n_sources":"n_sources_prot"}), on="model_id", how="left"))
    base["lineage"] = base.model_id.map(lm)
    return base, anchor

In [ ]:
def correlated_error_sensitivity(con, genes, rhos=(0.0, 0.25, 0.5), N=500, k=10, seed=1):
    """Test whether the independent-error assumption changes the win-rate ranking.

    For each gene, runs `monte_carlo_ci_corr` at each error correlation in `rhos` and
    compares the resulting top-1 win-rate ranking against the independent (rho=0)
    baseline by Spearman correlation, also tracking the largest single-line win-rate
    shift. A summary table (mean/min Spearman and mean max shift per rho) is printed;
    Spearman near 1 with small shifts indicates the independence assumption is
    defensible. Genes that fail to load are skipped with a message.

    Parameters
    ----------
    con : database connection
        Open warehouse connection.
    genes : iterable of str
        Genes (symbols or Ensembl IDs) to sweep.
    rhos : tuple of float, optional
        Error correlations to test; must include 0.0 as the baseline. Default (0.0, 0.25, 0.5).
    N : int, optional
        Monte-Carlo draws per run (default 500).
    k : int, optional
        Top-k cutoff passed through to the simulation (default 10).
    seed : int, optional
        RNG seed, shared across rhos so draws are comparable (default 1).

    Returns
    -------
    pandas.DataFrame
        One row per (gene, non-zero rho) with columns "gene", "rho_err",
        "spearman_vs_indep" and "max_winrate_shift_pp".
    """
    rows = []
    for gid in genes:
        try:
            base, anchor = _base_anchor(con, gid)
        except Exception as e:
            print("skip", gid, str(e)[:50]); continue
        base_wr = None
        for rho in rhos:
            res = monte_carlo_ci_corr(base, anchor, N=N, k=k, seed=seed, rho_err=rho)
            wr = res.set_index("model_id")["top1_winrate"]
            if rho == 0.0:
                base_wr = wr
            else:
                rho_s = base_wr.corr(wr.reindex(base_wr.index), method="spearman")
                # also track the largest single-line win-rate shift, in percentage points
                max_shift = (base_wr - wr.reindex(base_wr.index)).abs().max()
                rows.append({"gene": gid, "rho_err": rho,
                             "spearman_vs_indep": rho_s, "max_winrate_shift_pp": max_shift})
    df = pd.DataFrame(rows)
    print("Win-rate ranking stability vs the independent-error baseline:\n")
    print(df.groupby("rho_err").agg(
        mean_spearman=("spearman_vs_indep", "mean"),
        min_spearman=("spearman_vs_indep", "min"),
        mean_max_shift_pp=("max_winrate_shift_pp", "mean"),
    ).round(3).to_string())
    print("\n  Spearman ~1 and small win-rate shifts => correlated measurement errors")
    print("  barely change the ranking, so the independence assumption is defensible.")
    return df


# --- run it ---
core = pd.read_parquet(CORE_SCORE); core["ensg_id"] = core.ensg_id.str.upper()
genes = list(np.random.default_rng(0).choice(core.ensg_id.unique(), 20, replace=False))
corr_df = correlated_error_sensitivity(con, genes, rhos=(0.0, 0.25, 0.5), N=400)

In [ ]:
import ci_checks
genes = ci_checks._sample_genes(CORE_SCORE, n=30)
ci_checks.centre_draw_identity(con, run_gene_ci, _resolve_gid, CORE_SCORE)
ci_checks.order_consistency(con, genes, run_gene_ci)
ci_checks.knob_sweep(con, genes, run_gene_ci, ns=globals())          # ns=globals() so knobs can be perturbed
ctx = dict(_gene_leaves=_gene_leaves, build_anchor=build_anchor,
           core_score_chain=core_score_chain, _lookup=_lookup, CORE_SCORE=CORE_SCORE,
           RNA_WINSOR_BOUND=RNA_WINSOR_BOUND, PROT_STD_WINSOR_BOUND=PROT_STD_WINSOR_BOUND,
           REGIME3_SIG_THRESH=REGIME3_SIG_THRESH)
ci_checks.regime3_flip(con, genes, ctx)
ci_checks.correlated_error_sensitivity(con, genes, ctx)

## 5 · Verification and robustness

The method was validated by six checks. The code for each is in `ci_checks.py` (run against
the real warehouse over a 30-gene sample); the headline results, obtained on the real pipeline, are
summarised here so the notebook records them alongside the method.

| Check | What it tests | Result (real data) |
|---|---|---|
| **Centre-draw identity** | recompute equals the stored `core_score` | max\|Δ\| ≈ 1.8×10⁻⁸ (EGFR, TP53, MYC) |
| **Rank-calibration** | disjoint-CI pairs keep their order across draws | 100% of 570 pairs order-stable at ≥95% |
| **Self-consistency** | draw median vs stored centre, by score bin | mean \|median−centre\| = 0.006, saturation-shaped |
| **Convergence** | estimates stable as N grows | win-rate change ≤ 0.9 pp from N=10³ to 10⁴ |
| **Knob robustness** | win-rate ranking under ±50% knob moves | mean Spearman 0.91 (most sensitive: RNA base-floor) |
| **Variance decomposition** | RNA vs protein share of score variance | two-layer lines: protein ≈ 92% (±5%), RNA ≈ 9% |

Two structural findings emerged. First, interval **width is governed chiefly by score position**
(intervals are widest near the mid-range and compress near the 0/1 bounds); widening the winsor bounds
from ±5 to ±10 left this unchanged, indicating it is a property of the $\Phi$ transform rather than the
clip. Second, the regime substitution's membership **flips for ≈ 67% of two-layer lines** across
draws, which is why protein — entering through the volatile residual and the regime threshold —
dominates the score variance even though RNA drives the point score's discrimination. These are
complementary: saturation governs *how wide* an interval is; protein perturbation governs *what drives*
its variance, for lines that have protein. RNA-only lines derive their uncertainty entirely from RNA.

## 7 · Interpretation and scope

The intervals and stability figures are a statement of **reproducibility** — how much a recommendation
moves under realistic perturbation of its own inputs — and **not** a calibrated probability that a
line is biologically the most suitable model, since the procedure references only the pipeline's own
data and no external ground truth. Whether the confidence figures agree with external biological
measures (e.g. dependency or drug-response data) is a separate question and a natural next step; the
`Validation/eval.py` held-out evaluation provides a harness for it.